# Huấn luyện Full WP03 v2 (SOTA Peak-Capture & Multi-Horizon Ablation) trên Google Colab GPU

**Nhánh Git:** `fix/BPM25_WP03_sota-peak-capture` (`https://github.com/Duy-DS/Deep_Learning_Group.git`)  
**Phạm vi:** Pre-training 12 trạm × 3 seeds (`42, 1024, 2026`) × 4 horizons (`H ∈ {24, 48, 96, 192}`) + Fine-tuning 12 trạm + Toàn bộ Ablation RQ1/RQ2 (`Ablation 1, 2, 3`).

### Hướng dẫn nhanh (3 bước)
1. Trên menu Colab chọn **Thời gian chạy (Runtime)** → **Thay đổi loại thời gian chạy (Change runtime type)** → chọn **GPU L4** *(khuyên dùng)* hoặc **A100 / T4** → bấm **Lưu**.
2. Chạy lần lượt từ **Ô 1 → Ô 4 (Sanity Gate ~4 phút)**. Kiểm tra bảng `H=24` ở Ô 4 đảm bảo `PatchTST` / `Informer` vượt `DLinear` và `range_ratio_mean >= 0.45`.
3. Chạy tiếp **Ô 5 → Ô 7** (tự động sao lưu Google Drive sau từng chặng và tự resume nếu lỡ mất kết nối). Khi hoàn tất, gửi link file `wp03_v2_outputs.zip` trên Google Drive cho nhóm (**không** `git push` tệp `.pt` lên GitHub) và bấm **Ngắt kết nối và xóa thời gian chạy**.

## Ô 1 — Kiểm tra GPU & Kết nối Google Drive

In [ ]:
import os
import torch
from google.colab import drive

# 1. Kết nối Google Drive để lưu an toàn checkpoints & báo cáo
drive.mount('/content/drive')

DRIVE_BACKUP_DIR = '/content/drive/MyDrive/DL_Sgu_WP03_v2_outputs'
os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
print(f'[OK] Thư mục sao lưu Google Drive: {DRIVE_BACKUP_DIR}')

# 2. Xác nhận GPU CUDA đã được bật
if not torch.cuda.is_available():
    raise RuntimeError(
        'CHƯA BẬT GPU! Hãy vào menu: Thời gian chạy -> Thay đổi loại thời gian chạy -> chọn GPU (L4/T4/A100) rồi chạy lại từ Ô 1.'
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f'[OK] Đang dùng GPU: {gpu_name} ({vram_gb:.1f} GB VRAM) | PyTorch {torch.__version__}')
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## Ô 2 — Clone nhánh `fix/BPM25_WP03_sota-peak-capture` & Tải dữ liệu Git LFS

- Nếu repo là **Public**: chạy thẳng không cần làm gì thêm.
- Nếu repo là **Private**: thêm bí mật tên `GITHUB_TOKEN` trong mục **Mã bí mật (🔑 Secrets)** bên trái Colab *(hoặc ô code sẽ hỏi nhập Personal Access Token khi clone)*.

In [ ]:
import getpass
import os
import subprocess
from pathlib import Path

BRANCH = 'fix/BPM25_WP03_sota-peak-capture'
REPO_SLUG = 'Duy-DS/Deep_Learning_Group.git'
REPO_DIR = Path('/content/DL-Sgu')
PROJECT_DIR = REPO_DIR / 'course_project'

# Cài đặt & kích hoạt git-lfs
subprocess.run(['apt-get', '-qq', 'update'], check=True)
subprocess.run(['apt-get', '-qq', 'install', '-y', 'git-lfs'], check=True)
subprocess.run(['git', 'lfs', 'install'], check=True)

# Thử lấy GITHUB_TOKEN từ Colab Secrets nếu người dùng đã cấu hình
token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None

if REPO_DIR.exists():
    subprocess.run(['rm', '-rf', str(REPO_DIR)], check=True)

def clone_repo(auth_token: str | None) -> int:
    url = f'https://{auth_token}@github.com/{REPO_SLUG}' if auth_token else f'https://github.com/{REPO_SLUG}'
    env = dict(os.environ, GIT_TERMINAL_PROMPT='0')
    res = subprocess.run(
        ['git', 'clone', '--branch', BRANCH, '--single-branch', url, str(REPO_DIR)],
        env=env,
        capture_output=True,
        text=True,
    )
    if res.returncode != 0:
        print(res.stderr.strip())
    return res.returncode

rc = clone_repo(token)
if rc != 0:
    print('[LƯU Ý] Repo yêu cầu xác thực (Private). Nhập GitHub Personal Access Token (PAT) của bạn:')
    token = getpass.getpass('GitHub Token (PAT): ').strip()
    rc = clone_repo(token)
    if rc != 0:
        raise RuntimeError('Clone thất bại! Kiểm tra lại quyền truy cập repo hoặc GITHUB_TOKEN.')

os.chdir(PROJECT_DIR)
subprocess.run(['git', 'lfs', 'pull'], check=True)

# Xác minh tệp dữ liệu WP01 đã tải thật (không phải pointer 134 bytes)
train_csv = PROJECT_DIR / 'data' / 'preprocess' / 'train.csv'
if not train_csv.exists():
    raise FileNotFoundError(f'Không tìm thấy {train_csv}')
size_mb = train_csv.stat().st_size / (1024 ** 2)
print(f'[OK] Commit hiện tại:')
!git log -n 1 --oneline
print(f'[OK] Kích thước data/preprocess/train.csv = {size_mb:.2f} MB')
if size_mb < 100.0:
    raise RuntimeError(
        f'CẢNH BÁO: train.csv chỉ nặng {size_mb:.4f} MB (vẫn là Git LFS pointer!). '
        'Hãy kiểm tra lại git lfs pull hoặc quota Git LFS của repo.'
    )
!ls -lh data/preprocess/

## Ô 3 — Cài gói phụ thuộc & Chạy kiểm thử hồi quy (55/55 tests)

In [ ]:
%%bash
set -euo pipefail
cd /content/DL-Sgu/course_project
pip install -q pydantic pyyaml joblib scikit-learn pandas numpy tabulate onnx onnxruntime pytest
python -m pytest tests/test_wp03_sota_fixes.py -q

## Ô 4 — Sanity Gate 1 seed ở `H=24` (~3–5 phút trên GPU)

Chạy nhanh 1 seed (`seed=42`) trên toàn bộ 12 trạm ở `H=24` để xác minh trước khi chạy full:
- `patchtst` và `informer` có `mse_mean` / `mae_mean` **thấp hơn `dlinear`**.
- `range_ratio_mean >= 0.45` (không còn phẳng đường dự báo).
- Gửi ảnh bảng kết quả ô này vào nhóm xác nhận trước khi bấm Ô 5.

In [ ]:
import pandas as pd
import subprocess
from pathlib import Path
from IPython.display import display

PROJECT_DIR = Path('/content/DL-Sgu/course_project')
cmd = [
    'python', 'training_loop.py',
    '--config', 'configs/default.yaml',
    '--mode', 'multivariate',
    '--horizon', '24',
    '--seeds', '42',
]
subprocess.run(cmd, cwd=PROJECT_DIR, check=True)

summary_csv = PROJECT_DIR / 'outputs' / 'reports' / 'global_pretrain_summary_multivariate_H24.csv'
df_gate = pd.read_csv(summary_csv)
show_cols = [
    c for c in [
        'model', 'mse_mean', 'rmse_mean', 'mae_mean',
        'range_ratio_mean', 'spike_mae_ugm3_mean', 'peak_capture_ratio_mean', 'train_time_mean_sec'
    ] if c in df_gate.columns
]
print('=== KẾT QUẢ SANITY GATE (12 TRẠM, MULTIVARIATE, H=24, SEED=42) ===')
display(df_gate[show_cols])

# Sao lưu kết quả sanity gate lên Drive
!rsync -ah /content/DL-Sgu/course_project/outputs/reports/ /content/drive/MyDrive/DL_Sgu_WP03_v2_outputs/sanity_gate_reports/

## Ô 5 — Chạy Full 3 seeds × 4 horizons + Fine-tune 12 trạm + Toàn bộ Ablation RQ1/RQ2

- **An toàn chống mất điện / rớt mạng:** Chạy tuần tự 3 chặng (`ablation1` → `ablation2` → `ablation3`) và tự động đồng bộ toàn bộ `outputs/` lên Google Drive sau **mỗi chặng**.
- **Tự động Resume:** Nếu lỡ bị ngắt kết nối giữa chừng, chỉ cần chạy lại Ô 1, 2, 3 rồi chạy lại Ô 5 (với `FORCE_RETRAIN_ALL = False`) — hệ thống sẽ tự khôi phục phần đã lưu từ Google Drive và chỉ chạy nốt các phần chưa xong.
- Thời gian ước tính: **~40–60 phút (GPU L4)** | **~20–30 phút (GPU A100)** | **~1.5–2 giờ (GPU T4)**.

In [ ]:
import pandas as pd
import subprocess
import time
from pathlib import Path

PROJECT_DIR = Path('/content/DL-Sgu/course_project')
OUTPUTS_DIR = PROJECT_DIR / 'outputs'
DRIVE_BACKUP_DIR = Path('/content/drive/MyDrive/DL_Sgu_WP03_v2_outputs')

RUN_WITH_FINETUNE = True      # True = chạy kèm Fine-tune 12 trạm cho từng cấu hình
FORCE_RETRAIN_ALL = False     # False = tự resume từ các cấu hình đủ 3 seeds đã lưu trên Drive

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

# 1. Khôi phục kết quả đã có từ Google Drive (nếu đang resume phiên trước)
if not FORCE_RETRAIN_ALL and (DRIVE_BACKUP_DIR / 'reports').exists():
    print('[RESUME] Đang khôi phục checkpoints & reports từ Google Drive...')
    subprocess.run(
        ['rsync', '-ah', '--exclude', 'sanity_gate_reports', f'{DRIVE_BACKUP_DIR}/', f'{OUTPUTS_DIR}/'],
        check=True,
    )

# 2. Dọn riêng file summary H=24 nếu chỉ mới chạy 1 seed ở Ô 4 Sanity Gate (để chạy đủ 3 seeds)
h24_summary = OUTPUTS_DIR / 'reports' / 'global_pretrain_summary_multivariate_H24.csv'
if h24_summary.exists():
    df_chk = pd.read_csv(h24_summary)
    if 'num_seeds' in df_chk.columns and (df_chk['num_seeds'] < 3).any():
        print('[INFO] Phát hiện file H=24 từ Sanity Gate (1 seed) -> xóa để huấn luyện đầy đủ 3 seeds.')
        h24_summary.unlink()
        h24_runs = OUTPUTS_DIR / 'reports' / 'global_pretrain_runs_multivariate_H24.csv'
        if h24_runs.exists():
            h24_runs.unlink()

def sync_to_drive(tag: str) -> None:
    print(f'\n[BACKUP -> DRIVE] Đang đồng bộ outputs/ sau chặng: {tag}...')
    subprocess.run(
        ['rsync', '-ah', '--exclude', 'sanity_gate_reports', f'{OUTPUTS_DIR}/', f'{DRIVE_BACKUP_DIR}/'],
        check=True,
    )
    print(f'[BACKUP OK] Đã lưu an toàn vào {DRIVE_BACKUP_DIR}')

extra_flags = []
if FORCE_RETRAIN_ALL:
    extra_flags.append('--no-skip-existing')
if RUN_WITH_FINETUNE:
    extra_flags.append('--with-finetune')

t_all = time.perf_counter()
for stage in ['ablation1', 'ablation2', 'ablation3']:
    t0 = time.perf_counter()
    print('\n' + '=' * 80)
    print(f'>>> BẮT ĐẦU CHẶNG: {stage.upper()} | Flags: {extra_flags}')
    print('=' * 80)
    cmd = ['python', 'run_ablation.py', '--config', 'configs/default.yaml', '--ablations', stage] + extra_flags
    subprocess.run(cmd, cwd=PROJECT_DIR, check=True)
    print(f'[XONG {stage.upper()}] Thời gian: {(time.perf_counter() - t0) / 60:.1f} phút')
    sync_to_drive(stage)

# Tổng hợp lại một lần cuối cho trọn bộ báo cáo Markdown (skip_existing=True nên chạy mất ~2 giây)
cmd_final = ['python', 'run_ablation.py', '--config', 'configs/default.yaml', '--ablations', 'all']
if RUN_WITH_FINETUNE:
    cmd_final.append('--with-finetune')
subprocess.run(cmd_final, cwd=PROJECT_DIR, check=True)
sync_to_drive('FINAL_ALL')

print(f'\n[HOÀN TẤT TOÀN BỘ] Tổng thời gian: {(time.perf_counter() - t_all) / 60:.1f} phút')

## Ô 6 — Xem báo cáo Ablation & Kiểm tra đủ toàn bộ Checkpoint `.pt`

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display

PROJECT_DIR = Path('/content/DL-Sgu/course_project')
OUTPUTS_DIR = PROJECT_DIR / 'outputs'

global_pts = sorted((OUTPUTS_DIR / 'checkpoints' / 'global').glob('*.pt'))
station_pts = sorted((OUTPUTS_DIR / 'checkpoints' / 'stations').rglob('*.pt'))
scalers = sorted((OUTPUTS_DIR / 'checkpoints' / 'global').glob('*.joblib'))

print(f'[CHECKPOINT GLOBAL]  {len(global_pts)} tệp .pt')
print(f'[CHECKPOINT STATION] {len(station_pts)} tệp .pt')
print(f'[SCALERS]            {[p.name for p in scalers]}')

md_report = OUTPUTS_DIR / 'ablation' / 'ablation_report_summary.md'
if md_report.exists():
    display(Markdown(md_report.read_text(encoding='utf-8')))
else:
    print('[CẢNH BÁO] Chưa thấy ablation_report_summary.md')

## Ô 7 — Đóng gói `wp03_v2_outputs.zip` lên Google Drive để gửi cho nhóm

> Sau khi chạy xong Ô 7, vào Google Drive chia sẻ link tệp **`wp03_v2_outputs.zip`** cho nhóm (bàn giao cho WP04 XAI & WP05 ONNX/Streamlit), rồi bấm **Thời gian chạy → Ngắt kết nối và xóa thời gian chạy** để dừng tính điểm GPU.

In [ ]:
%%bash
set -euo pipefail
cd /content/DL-Sgu/course_project
ZIP_PATH="/content/drive/MyDrive/wp03_v2_outputs.zip"
rm -f "${ZIP_PATH}"
zip -rq "${ZIP_PATH}" outputs/
ls -lh "${ZIP_PATH}"
echo "[HOÀN TẤT] Đã đóng gói thành công tại Google Drive: ${ZIP_PATH}"
echo "Hãy gửi link file này cho nhóm và bấm 'Ngắt kết nối và xóa thời gian chạy' trên Colab!"